In [2]:
from models.plm import slice_validation
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm 
import os
os.environ["TQDM_DISABLE"] = "0"  # keep bars but force text mode

from transformers import pipeline
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_and_split
from evaluate import append_results, build_part_b_rows, sarcasm_f1
from models.plm import (
    HYPOTHESES,
    NLI_MODEL,
    SENTIMENT_MODEL,
    collect_part_b,
    hypothesis_frame,
    load_nli_pipeline,
    load_sentiment_pipeline,
    slice_validation,
)

from train import DEFAULT_SEED, set_seed

pd.set_option("display.max_colwidth", 160)
set_seed(DEFAULT_SEED)

LIMIT = None  # entier : essai court, results.csv n'est pas modifié
BATCH_SIZE = 8

bundle = load_and_split()
val_data = slice_validation(bundle["val"], LIMIT)
y_val = val_data["sarcasm"].astype(bool)


import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

# ==============================
# GPU CONFIGURATION
# ==============================

MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8
MAX_LENGTH = 128

print("Device:", DEVICE)

# ==============================
# LOAD MODEL
# ==============================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32
).to(DEVICE)

model.eval()

print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

# ==============================
# PREPARE DATA
# ==============================

texts = val_data["tweet"].fillna("").astype(str).tolist()

def encode_label(x):
    if isinstance(x, str):
        x = x.strip().lower()
        if x in ("true", "1"):
            return 1
        if x in ("false", "0"):
            return 0
    return int(x)

y_true = val_data["sarcasm"].map(encode_label).to_numpy()

configs = {
    "Arabic": [
        "هذه التغريدة ساخرة.",
        "هذه التغريدة جادة."
    ],
    "French": [
        "Ce tweet est sarcastique.",
        "Ce tweet est sérieux."
    ],
    "English": [
        "This tweet is sarcastic.",
        "This tweet is serious."
    ]
}

# ==============================
# GPU BATCH INFERENCE
# ==============================

@torch.inference_mode()
def predict_nli(texts, hypotheses):

    probabilities = []

    for start in tqdm(range(0, len(texts), BATCH_SIZE)):

        batch = texts[start:start + BATCH_SIZE]

        # Two hypothesis pairs per tweet
        premises = [
            text
            for text in batch
            for _ in hypotheses
        ]

        hypotheses_batch = hypotheses * len(batch)

        inputs = tokenizer(
            premises,
            hypotheses_batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)

        logits = model(**inputs).logits

        # Entailment probabilities
        entailment_id = next(
            int(i) for i, label in model.config.id2label.items()
            if label.lower() == "entailment"
        )

        # Match Hugging Face single-label zero-shot scoring:
        # softmax entailment logits across candidate labels
        entailment_logits = logits[:, entailment_id]
        scores = entailment_logits.reshape(-1, len(hypotheses))
        probs = torch.softmax(scores.float(), dim=1)[:, 0]

        probabilities.extend(probs.cpu().numpy())

    return np.array(probabilities)

# ==============================
# THRESHOLD OPTIMIZATION
# ==============================

def optimize_threshold(y_true, probs):

    thresholds = np.arange(0.05, 0.96, 0.01)

    f1_scores = [
        f1_score(y_true, probs >= t, zero_division=0)
        for t in thresholds
    ]

    best_idx = np.argmax(f1_scores)

    return thresholds[best_idx], f1_scores[best_idx]

# ==============================
# RUN EXPERIMENTS
# ==============================

results = []

for language, hypotheses in configs.items():

    print(f"\nRunning {language} zero-shot NLI...")

    probs = predict_nli(texts, hypotheses)

    threshold, best_f1 = optimize_threshold(
        y_true, probs
    )

    preds = (probs >= threshold).astype(int)

    results.append({
        "Language": language,
        "Threshold": round(float(threshold), 3),
        "F1": round(float(best_f1), 4),
        "Precision": round(float(precision_score(y_true, preds, zero_division=0)), 4),
        "Recall": round(float(recall_score(y_true, preds, zero_division=0)), 4),
        "PR_AUC": round(float(average_precision_score(y_true, probs)), 4)
    })

results_df = pd.DataFrame(results).sort_values(
    "F1", ascending=False
)

print("\nFINAL RESULTS")
print(results_df.to_string(index=False))


Device: cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

GPU: CPU

Running Arabic zero-shot NLI...


  0%|          | 0/157 [00:00<?, ?it/s]


Running French zero-shot NLI...


  0%|          | 0/157 [00:00<?, ?it/s]


Running English zero-shot NLI...


  0%|          | 0/157 [00:00<?, ?it/s]


FINAL RESULTS
Language  Threshold     F1  Precision  Recall  PR_AUC
  French       0.75 0.4130     0.2929  0.7005  0.3493
 English       0.82 0.4030     0.3002  0.6129  0.3399
  Arabic       0.82 0.3896     0.2920  0.5853  0.3389
